# 3. Connect the agent and check its behaviour

Retrieve the approved global ink report from the deployed nodes. Then check
unsupported arguments, repeated calls and unavailable evidence.

Edit `work/paris-federation-agent/federation_agent/guided.py`. Complete its
tool schema, argument validation and result cache. The collector is provided.
The [agent guide](../docs/agent.md) describes the contract and configuration.

Download the agent starter once it is released; the MNIST exercises
use the earlier app release.


In [ ]:
from pathlib import Path
import sys

ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "notebooks/workshop.py").is_file() and (path / "configs/train.toml").is_file()
)
sys.path.insert(0, str(ROOT / "notebooks"))
from workshop import guided_agent, replay_agent


## Define the capability

The tool is named `get_federation_summary`. It accepts an empty JSON object
and no extra properties. Its result contains one approved global report or
a generic error. The model cannot choose nodes, request examples or query an
arbitrary subset.

Complete the schema in `guided.py`, save the file and run this check.


In [ ]:
guided = guided_agent(ROOT)
assert len(guided.TOOLS) == 1
schema = guided.TOOLS[0]
assert schema["type"] == "function"
assert schema["name"] == "get_federation_summary" and schema["strict"] is True
parameters = schema["parameters"]
assert parameters["type"] == "object" and parameters["properties"] == {}
assert parameters["required"] == [] and parameters["additionalProperties"] is False
print("Your tool schema passed")


## Finish dispatch and try the local checks

Validate the exact name and empty-object arguments before collection. Reuse
the collected `approved_aggregate`, or a generic failure, during one agent run.
Save your edits, then run the cell below. It launches a fresh Python process
using your guided code, synthetic node replies and a scripted model. No hosted
or model request is made by this local check.


In [ ]:
complete = replay_agent(ROOT, scenario="complete", repeat=2)
assert complete["tool_calls"] == 2 and complete["grid_calls"] == 3
print("Two tool calls reused one synthetic collection")


## Query the real federation

Update the **existing** entries under `[tool.flwr.app.config]` in your
downloaded agent's `pyproject.toml`:

```toml
summary-source = "analytics"
release-id = "analytics-moments-v1"
cohort-id = "paris-mnist-three"
cohort-node-ids = "NODE_0,NODE_1,NODE_2"
implementation = "guided"
loop-implementation = "provided"
model-free = true
```

Replace the node placeholders with the three announced decimal node IDs, without
spaces. Use the announced release ID if it differs. These settings select the
global ink report used for this exercise.

In a terminal, from the guide root:

```sh
cd work/paris-federation-agent
flwr chat
```

Inside Chat:

```text
/federation @charles/flta
/load .
Which digit has the highest mean ink? Include its count, standard deviation and source run.
```

`model-free = true` still queries the real nodes, returning checked JSON.
Wait for the model settings before enabling a natural-language answer.
Reload with `/load .` after changing code or configuration.

**Checkpoint:** your modified agent retrieves an approved report from the
three deployed nodes, with the analytics source run preserved. Keep that
successful trace before continuing with local failure cases.


## Reject unsupported calls before collection

Save your dispatcher edits and rerun the next cell. The spy raises if an
invalid request reaches the Grid. Add another unsupported case of your own.


In [ ]:
guided = guided_agent(ROOT)
from federation_agent.replay import FIXTURE_CONFIG, FixtureGrid

fixture_cfg = dict(FIXTURE_CONFIG, **{
    "summary-source": "analytics", "release-id": "analytics-moments-v1"
})

class NoNetwork:
    def __init__(self):
        self.calls = 0

    def call(self, request):
        self.calls += 1
        raise AssertionError("Invalid request reached the Grid")

bad_requests = [
    ("unknown_tool", "{}"),
    ("get_federation_summary", '{"node_id": "123"}'),
    ("get_federation_summary", "[1, 2]"),
    ("get_federation_summary", "{broken"),
    ("get_federation_summary", "null"),
]
for name, arguments in bad_requests:
    spy = NoNetwork()
    result = guided.SummaryTool(spy, fixture_cfg).dispatch(name, arguments)
    assert result.get("error") and "approved_aggregate" not in result
    assert spy.calls == 0
print("Unsupported requests performed zero Grid operations")


## Cache both success and failure

Each synthetic collection uses discovery, send and receive: three Grid calls.
Calling the same tool again during that agent run must reuse its success or
failure. A new Chat prompt may collect again.

Missing, conflicting and wrong-source evidence must not produce a partial
approved report. The tool should expose a generic error, without local paths
or raw transport replies.


In [ ]:
guided = guided_agent(ROOT)
for scenario in ["complete", "missing", "conflicting", "wrong-source"]:
    grid = FixtureGrid(scenario, source="analytics")
    tool = guided.SummaryTool(grid, fixture_cfg)
    first = tool.dispatch("get_federation_summary", "{}")
    second = tool.dispatch("get_federation_summary", "{}")
    assert first == second and len(grid.calls) == 3
    if scenario == "complete":
        assert "approved_aggregate" in first and "error" not in first
    else:
        assert first.get("error") and "approved_aggregate" not in first
    print(f"{scenario}: repeated call reused one collection")


In [ ]:
missing = replay_agent(ROOT, scenario="missing", repeat=2)
assert missing["tool_calls"] == 2 and missing["grid_calls"] == 3


## Keep your observations

Reload the agent and repeat your working query on the deployed federation
after your changes. Local synthetic results alone do not complete that checkpoint.

| Observation | Your result |
| --- | --- |
| Approved release and analytics source run | |
| Agent run or trace identifier | |
| Highest-mean-ink digit, count and standard deviation | |
| Model-free or model-explained run | |
| An unsupported call and its Grid-call count | |
| Repeated-success and repeated-failure checks | |

Why do three agreeing reports describe one 6,000-image population rather
than three populations to add together? What should the agent say when one
of those reports is missing?

**Checkpoint:** demonstrate working deployed retrieval, rejection before
collection, and cached success and failure. Keep the source report's population
clear: the 6,000 training images used for our ink statistics.

If you finish early, choose an [agent extension](../docs/extensions.md#after-the-agent)
or [connect your own SuperNode](../docs/extensions.md#connect-your-own-supernode)
in the playground federation once access and partition details are confirmed.
